# Inference Data Processing Walkthrough
---

-  This Section provides a walkthrough of the Inference acquisition pipeline demonstrated 
### 1. Setup and Import Libraries

In [1]:
import joblib
import numpy as np
from pyproj import Geod
import shapely.geometry
import folium
import pandas as pd
import os
import geopandas as gpd
from datetime import datetime
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from new_utils import gdf_from_geojson
from new_app import mask_downloaded_image, convert_mask_image_to_gdf, get_any_image_from_sentinelhub
from IPython.display import display, Markdown
from data_downloader import (
    states_gdf_from_geojson, 
    get_available_dates, 
    get_dictionary_of_images_from_evalscripts, 
    get_total_polygon_from_gdf
)
from data_inference_collector import (
    get_square_list_for_state, 
    convert_square_to_polygon, 
    calculate_area_in_square_meters
)


def map_cols(gdf):
    data = gdf.copy()
    month_mapping = {
        "June": 7,
        "July": 8,
        "August": 9,
        "September": 10,
        "October": 11
    }
    rename_dict = {}
    for month, suffix in month_mapping.items():
        b8 = data[f"{month}_band_8"]
        b4 = data[f"{month}_band_4"]
        data[f"ndvi_{suffix}"] = (b8 - b4) / (b8 + b4)
        rename_dict.update({
            f"{month}_band_1":  f"B1_{suffix}",
            f"{month}_band_10": f"B10_{suffix}",
            f"{month}_band_11": f"B11_{suffix}",
            f"{month}_band_12": f"B12_{suffix}",
            f"{month}_band_2":  f"B2_{suffix}",
            f"{month}_band_3":  f"B3_{suffix}",
            f"{month}_band_4":  f"B4_{suffix}",
            f"{month}_band_5":  f"B5_{suffix}",
            f"{month}_band_6":  f"B6_{suffix}",
            f"{month}_band_7":  f"B7_{suffix}",
            f"{month}_band_8":  f"B8_{suffix}",
            f"{month}_band_9":  f"B8A_{suffix}",
            f"{month}_band_13": f"B9_{suffix}",
            f"{month}_FCOVER":  f"fcover_{suffix}"
        })
    data = data.rename(columns=rename_dict)
    new_order = []
    for suffix in sorted(month_mapping.values()):
        new_order.extend([
            f"B1_{suffix}", f"B10_{suffix}", f"B11_{suffix}", f"B12_{suffix}",
            f"B2_{suffix}", f"B3_{suffix}", f"B4_{suffix}", f"B5_{suffix}",
            f"B6_{suffix}", f"B7_{suffix}", f"B8_{suffix}", f"B8A_{suffix}",
            f"B9_{suffix}", f"ndvi_{suffix}", f"fcover_{suffix}"
        ])
    new_order.extend(["location_name", "geometry"])
    data = data[new_order]
    return data


def get_month_name(date_str):
    return datetime.strptime(date_str, "%Y-%m-%d").strftime("%B")

def combine_months_geojsons(month_geojson_dict, evalscript=None):
    gdfs = []
    for i, (month, geojson_path) in enumerate(month_geojson_dict.items()):
        gdf = gdf_from_geojson(geojson_path=geojson_path, crs="EPSG:4326")
        for col in ['date', 'evalscript']:
            if col in gdf.columns:
                gdf = gdf.drop(columns=col)
        band_cols = [col for col in gdf.columns if 'band' in col]
        if evalscript == None:
            rename_dict = {col: f"{month}_{col}" for col in band_cols}
        else:
            rename_dict = {col: f"{month}_{evalscript}" for col in band_cols}

        gdf = gdf.rename(columns=rename_dict)
        if i > 0:
            if "location_name" in gdf.columns:
                gdf = gdf.drop(columns="location_name")
            if "geometry" in gdf.columns:
                gdf = gdf.drop(columns="geometry")
        gdfs.append(gdf)
    combined_gdf = pd.concat(gdfs, axis=1)
    return combined_gdf

def display_map(gdf, title, column=None, m=None, opacity=0.6):
    if len(gdf) > 1000:
        gdf = gdf.sample(1000)

    if m is None:
        centroid = gdf.geometry.centroid.iloc[0]
        m = folium.Map(location=[centroid.y, centroid.x], zoom_start=10)

    gdf.explore(column=column, style_kwds={"fillOpacity": opacity}, m=m)
    
    display(Markdown(f"### {title}"))
    display(m)

    return m

def get_bbox_info(gdf, verbose=False):
    geod = Geod(ellps="WGS84")
    bbox = gdf.total_bounds
    polygon = shapely.geometry.box(*bbox, ccw=True)
    area = abs(geod.geometry_area_perimeter(polygon)[0])
    perimeter = abs(geod.geometry_area_perimeter(polygon)[1])
    width_line_coords = [(bbox[1], bbox[0]), (bbox[1], bbox[2])]
    width_line = shapely.geometry.LineString(width_line_coords)
    width = abs(geod.geometry_area_perimeter(width_line)[1])
    height_line_coords = [(bbox[1], bbox[0]), (bbox[3], bbox[0])]
    height_line = shapely.geometry.LineString(height_line_coords)
    height = abs(geod.geometry_area_perimeter(height_line)[1])
    gdf_bbox = gdf.total_bounds
    gdf_bbox_polygon = shapely.geometry.box(*gdf_bbox, ccw=True)
    gdf_bbox = [(gdf_bbox[1], gdf_bbox[0]), (gdf_bbox[3], gdf_bbox[2])]
    if verbose:
        print(f'Area of State: {area/1000000} km2')
        print(f'Perimeter of State: {perimeter/1000} km')
        print(f'Width of Bounding Box: {width/1000} km')
        print(f'Height of Bounding Box: {height/1000} km')
        print(f'Area of Bounding Box: {(calculate_area_in_square_meters(gdf_bbox_polygon))/1000000} km2')
        print(f'Perimeter of Bounding Box: {(2 * (width + height))/1000} km')
    return width, height, area, perimeter, gdf_bbox 

def get_same_month_dates(target_date, available_dates):
    target_month = datetime.strptime(target_date, "%Y-%m-%d").month
    target_year = datetime.strptime(target_date, "%Y-%m-%d").year
    same_month_dates = [
        date for date in available_dates
        if datetime.strptime(date, "%Y-%m-%d").month == target_month
        and datetime.strptime(date, "%Y-%m-%d").year == target_year
    ]
    return same_month_dates

def get_best_date_in_month_for_gdf(target_date, gdf, location_name=None, find_least_cloud_cover=True):
    year = target_date.split("-")[0]
    available_dates_year = get_available_dates(gdf, year)
    available_dates_month = get_same_month_dates(target_date, available_dates_year)
    if find_least_cloud_cover:
        location_name =  f'cloud_cover_{datetime.now().strftime("%Y%m%d_%H%M%S")}' if location_name is None else location_name
        clp_averages = []
        for date in tqdm(available_dates_month, "Calculating Cloud Cover ..."):
            clp, _ = get_any_image_from_sentinelhub(get_total_polygon_from_gdf(gdf), date, 'CLP', location_name)
            clp_averages.append(np.mean(clp))
        min_cloud_index = np.argmin(clp_averages)
        least_cloud_cover_date = available_dates_month[min_cloud_index]
        return least_cloud_cover_date
    else:
        return available_dates_month

def squares_list_to_gdf(squares_list, square_name):
    geom = [convert_square_to_polygon(square) for square in squares_list]
    gdf = gpd.GeoDataFrame(geometry=geom)
    gdf.crs = "EPSG:4326"
    gdf['square_id'] = [f'{square_name}_{i}' for i in range(len(gdf))]
    gdf['Area_M2'] = gdf['geometry'].apply(calculate_area_in_square_meters)
    gdf['Area_KM2'] = gdf['Area_M2']/1000000
    gdf['location'] = square_name
    return gdf

def get_geojsons_data_dict_by_month(total_polygon, mask_gdf, dates, location_name, evalscript):
    month_geojson_dict = {}
    for date in tqdm(dates, f"Processing dates for {location_name}"): # dates should be selected to align with how the model was trained and what imagery is available with lowest Cloud Cover
        if (not os.path.exists(f"data/satellite_images/{location_name}/{evalscript}/{date}")) or (not os.listdir(f"data/satellite_images/{location_name}/{evalscript}/{date}")):
            get_dictionary_of_images_from_evalscripts(total_polygon=total_polygon, date=date, location_name=location_name)
        mask_path = mask_downloaded_image(mask_gdf=mask_gdf, location_name=location_name, date=date, evalscript=evalscript)
        geojson_path = convert_mask_image_to_gdf(location_name=location_name, date=date, evalscript=evalscript, crs="EPSG:4326")
        month = get_month_name(date)
        month_geojson_dict[month] = geojson_path
        print(f"Saved output for {month} to {geojson_path}")
    return month_geojson_dict




In [2]:
# Inference Settings
STATE_NAME = "El Gazira"
BIG_SQUARE_SIZE = 20 #KM^2
INF_SQUARE_SIZE = 0.25 #KM^2
INF_NUMBER_OF_SQUARES = 2000
# INF_NUMBER_OF_SQUARES = 1
MONTHS_DATES = [
    '2024-06-01',
    '2024-07-01',
    '2024-08-01',
    '2024-09-01',
    '2024-10-01',
]

### 2. Prepare BIG squares and INF squares

In [3]:

state_gdf = states_gdf_from_geojson(file_path='./data/geojsons/sudan_states.geojson')
state_gdf = state_gdf[state_gdf["State"]==STATE_NAME].reset_index(drop=True)

big_squares = get_square_list_for_state(state_gdf, max_width=BIG_SQUARE_SIZE, max_height=BIG_SQUARE_SIZE)
inf_squares = get_square_list_for_state(state_gdf, max_width=INF_SQUARE_SIZE, max_height=INF_SQUARE_SIZE)

big_squares_gdf = squares_list_to_gdf(big_squares, square_name=f"{STATE_NAME} - BIG - {BIG_SQUARE_SIZE}")
inf_squares_gdf = squares_list_to_gdf(inf_squares, square_name=f"{STATE_NAME} - INF - {INF_SQUARE_SIZE}")


inf_squares_gdf = inf_squares_gdf.sample(INF_NUMBER_OF_SQUARES).reset_index(drop=True) # A seed must be set here to ensure  sampling the same every time
print(f"Loaded {len(big_squares)} big squares in GDF of shape {big_squares_gdf.shape}")
print(f"Loaded {len(inf_squares_gdf)} (out of {len(inf_squares)}) inf squares in GDF of shape {inf_squares_gdf.shape}")

2025-04-18 00:39:16.783 
  command:

    streamlit run /opt/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]
100%|██████████| 1470/1470 [00:01<00:00, 1439.08it/s]


Loaded 399 big squares in GDF of shape (399, 5)
Loaded 2000 (out of 2494590) inf squares in GDF of shape (2000, 5)


In [4]:
# from data_downloader import states_gdf_from_geojson
# state_gdf = states_gdf_from_geojson()
# state_gdf = state_gdf[state_gdf["State"]=="El Gazira"]
# gdf1 = inf_squares_gdf.copy()
# gdf2 = state_gdf.reset_index().copy()

# filtered_gdf = gpd.sjoin(gdf1, gdf2, how="inner", predicate="within")

# # If you want to remove duplicate columns from the join
# filtered_gdf = filtered_gdf.drop(columns=[col for col in filtered_gdf.columns if col.endswith('_right')])
# filtered_gdf.to_file("inf_random_sample.geojson", driver="GeoJSON")
# print("./inf_random_sample.geojson")


### 3. Get the date at each target month with the least cloud cover for the desired location

In [5]:
PRE_SELECTED_DATES = ['2024-06-15', '2024-07-25', '2024-08-09', '2024-09-23', '2024-10-13']
sample_square = big_squares_gdf.iloc[[0]] # The Big Square 25x25 KM. used to find the best cloud cover days. Ideally should be done per square
location_name = sample_square['square_id'][0]
if PRE_SELECTED_DATES is  None:
    pre_selected_dates = []
    for month_date in tqdm(MONTHS_DATES, "Processing Months ..."):
        pre_selected_dates.append(get_best_date_in_month_for_gdf(month_date, sample_square, location_name, find_least_cloud_cover=True))
else:
    pre_selected_dates = PRE_SELECTED_DATES
print(f"Selected Date for each month: {pre_selected_dates}")



Selected Date for each month: ['2024-06-15', '2024-07-25', '2024-08-09', '2024-09-23', '2024-10-13']


# 4. Download Imagery from big squares and map it to small squares across the stats for selected dates

In [6]:
# import concurrent.futures
# from tqdm.auto import tqdm
# import os

# def process_square_and_date(args):
#     """Process a single big square and date combination"""
#     big_square, date = args
#     location_name = big_square['square_id'].values[0]
#     total_polygon = get_total_polygon_from_gdf(gdf=big_square)
#     for data_type in ["FCOVER", "ALL"]:
#         if not os.path.exists(f"data/satellite_images/{location_name}/{data_type}/{date}"):
#             get_any_image_from_sentinelhub(total_polygon, date, data_type, location_name)

# def optimize_sentinel_processing(big_squares_gdf, pre_selected_dates, max_workers=8, chunk_size=10):
#     """Process big squares and dates efficiently using parallel processing and chunking"""
#     all_tasks = []
#     for big_square_index in tqdm(range(len(big_squares_gdf))):
#         current_big_square = big_squares_gdf.iloc[[big_square_index]]
#         for date in pre_selected_dates:
#             all_tasks.append((current_big_square, date))

#             location_name = current_big_square['square_id'].values[0]
#             total_polygon = get_total_polygon_from_gdf(gdf=current_big_square)
#             for data_type in ["FCOVER", "ALL"]:
#                 if not os.path.exists(f"data/satellite_images/{location_name}/{data_type}/{date}"):
#                     get_any_image_from_sentinelhub(total_polygon, date, data_type, location_name)

    
    # # Process in parallel with progress tracking
    # with tqdm(total=len(all_tasks), desc="Processing square-date combinations") as pbar:
        

    #     # Use process pool if tasks are CPU-bound, ThreadPool if I/O bound
    #     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
    #         # Submit tasks in chunks to avoid memory issues with very large datasets
    #         for i in range(0, len(all_tasks), chunk_size):
    #             chunk = all_tasks[i:i+chunk_size]
    #             chunk_futures = [executor.submit(process_square_and_date, task) for task in chunk]
                
    #             for future in concurrent.futures.as_completed(chunk_futures):
    #                 try:
    #                     future.result()
    #                 except Exception as e:
    #                     print(f"Error processing task: {e}")
    #                 finally:
    #                     pbar.update(1)
# inference_results = optimize_sentinel_processing(big_squares_gdf, pre_selected_dates)
    

In [7]:
# import multiprocessing
# from functools import partial
# from tqdm.auto import tqdm

# def process_big_square(big_square_index, big_squares_gdf, inf_squares_gdf, pre_selected_dates):
#     # Extract the current big square
#     current_big_square = big_squares_gdf.iloc[[big_square_index]]
#     location_name = current_big_square['square_id'].values[0]
#     total_polygon = get_total_polygon_from_gdf(gdf=current_big_square)
    
#     # Find small inference squares within the current big square
#     mask_inf_squares = current_big_square.clip(inf_squares_gdf)
#     if len(mask_inf_squares) == 0:
#         return None
    
#     # Process FCOVER data
#     month_geojson_dict_fcover = get_geojsons_data_dict_by_month(
#         total_polygon, mask_inf_squares, pre_selected_dates, location_name, evalscript="FCOVER"
#     )
#     clipped_gdf_fcover = combine_months_geojsons(month_geojson_dict_fcover, "FCOVER")
    
#     # Process all bands data
#     month_geojson_dict_all_bands = get_geojsons_data_dict_by_month(
#         total_polygon, mask_inf_squares, pre_selected_dates, location_name, evalscript="ALL"
#     )
#     clipped_gdf_all_bands = combine_months_geojsons(month_geojson_dict_all_bands)
    
#     # Combine the data
#     fcover_columns = [col for col in clipped_gdf_fcover.columns if 'FCOVER' in col]
#     clipped_gdf = clipped_gdf_all_bands.join(clipped_gdf_fcover[fcover_columns])
    
#     print(f"processed gdf of shape {clipped_gdf.shape} in {location_name}")
#     return clipped_gdf

# def main_parallel_processing(big_squares_gdf, inf_squares_gdf, pre_selected_dates, num_processes=None):
#     """
#     Parallel processing of big squares
    
#     Parameters:
#     -----------
#     big_squares_gdf : GeoDataFrame
#         GeoDataFrame containing the big squares (25x25 km)
#     inf_squares_gdf : GeoDataFrame
#         GeoDataFrame containing the small inference squares (2x2 km)
#     pre_selected_dates : list
#         List of pre-selected dates
#     num_processes : int, optional
#         Number of processes to use. If None, uses the number of CPUs.
    
#     Returns:
#     --------
#     list
#         List of GeoDataFrames, one for each processed big square
#     """
#     if num_processes is None:
#         num_processes = multiprocessing.cpu_count()
    
#     # Create a partial function with all arguments except big_square_index
#     process_func = partial(
#         process_big_square,
#         big_squares_gdf=big_squares_gdf,
#         inf_squares_gdf=inf_squares_gdf,
#         pre_selected_dates=pre_selected_dates
#     )
    
#     # Create a pool of workers
#     pool = multiprocessing.Pool(processes=num_processes)
    
#     # Process big squares in parallel with a progress bar
#     indices = range(len(big_squares_gdf))
#     results = list(tqdm(pool.imap(process_func, indices), 
#                         total=len(indices), 
#                         desc="Processing big squares in parallel"))
    
#     # Close the pool
#     pool.close()
#     pool.join()
    
#     # Filter out None results (squares with no inference squares)
#     inference_gdfs = [result for result in results if result is not None]
#     return inference_gdfs

# # Usage:
# inference_gdfs = main_parallel_processing(big_squares_gdf, inf_squares_gdf, pre_selected_dates)


In [8]:
inference_gdfs = []
for big_square_index in tqdm(range(len(big_squares_gdf)), "processing big squares"): # May Take around 1000 minutes without any optimization
    current_big_square = big_squares_gdf.iloc[[big_square_index]] # The Big Square 25x25 KM. Ideally pre-downloaded 
    location_name = current_big_square['square_id'].values[0]
    break
location_name

processing big squares:   0%|          | 0/399 [00:00<?, ?it/s]

'El Gazira - BIG - 20_0'

In [ ]:
inference_gdfs = []
for big_square_index in tqdm(range(len(big_squares_gdf)), "processing big squares"): # May Take around 1000 minutes without any optimization
    current_big_square = big_squares_gdf.iloc[[big_square_index]] # The Big Square 25x25 KM. Ideally pre-downloaded 
    location_name = current_big_square['square_id'].values[0]
    total_polygon = get_total_polygon_from_gdf(gdf=current_big_square)
    
    mask_inf_squares = current_big_square.clip(inf_squares_gdf) # The small inference squares 2x2 KM that are within the current big square
    if len(mask_inf_squares) == 0:
        continue

    month_geojson_dict_fcover = get_geojsons_data_dict_by_month(total_polygon, mask_inf_squares, pre_selected_dates, location_name, evalscript="FCOVER")
    clipped_gdf_fcover = combine_months_geojsons(month_geojson_dict_fcover, "FCOVER")

    month_geojson_dict_all_bands = get_geojsons_data_dict_by_month(total_polygon, mask_inf_squares, pre_selected_dates, location_name, evalscript="ALL")
    clipped_gdf_all_bands = combine_months_geojsons(month_geojson_dict_all_bands)

    fcover_columns = [col for col in clipped_gdf_fcover.columns if 'FCOVER' in col]
    clipped_gdf = clipped_gdf_all_bands.join(clipped_gdf_fcover[fcover_columns])
    print(f"processed gdf of shape {clipped_gdf.shape} in  {location_name}")
    inference_gdfs.append(clipped_gdf)
    break



### 5. Save The processed inference data

In [ ]:
inference_data_gdf = pd.concat(inference_gdfs, ignore_index=True)
inference_data_gdf = map_cols(inference_data_gdf)
# output_path = f'inference_data_gdf_{datetime.now().strftime("%Y%m%d_%H%M%S")}.joblib'
# joblib.dump(inference_data_gdf, output_path)
output_path = f'inference_data_gdf_{datetime.now().strftime("%Y%m%d_%H%M%S")}.parquet'
inference_data_gdf.to_parquet(output_path)

file_size_gb = os.path.getsize(output_path) / (1024 ** 3)
# Print shape, size, and save location
print(f"Final combined gdf shape: {inference_data_gdf.shape}")
print(f"File size: {file_size_gb:.5f} GB")
print(f"Saved to: {output_path}")


In [ ]:

output_path = f'inference_data_gdf_{datetime.now().strftime("%Y%m%d_%H%M%S")}.joblib'
joblib.dump(inference_data_gdf, output_path)
# output_path = f'inference_data_gdf_{datetime.now().strftime("%Y%m%d_%H%M%S")}.parquet'
# inference_data_gdf.to_parquet(output_path)

file_size_gb = os.path.getsize(output_path) / (1024 ** 3)
# Print shape, size, and save location
print(f"Final combined gdf shape: {inference_data_gdf.shape}")
print(f"File size: {file_size_gb:.5f} GB")
print(f"Saved to: {output_path}")


In [31]:
import joblib
output_path = f'inference_data_gdf_20250415_095827.joblib'

data = joblib.load(output_path)
joblib.dump(data, "inference_data_20204.joblib")
data.to_csv("inference_data_20204.csv", index_label="index")